# HW3: Deep Learning - VAE, XRD Sequence Models, and CrabNet (Spring 2026)


## Prerequisites: Environment Setup

Before you begin, set up your Python environment.

**Setup Steps:**
1.  Navigate to the `HW3` directory in your terminal.
2.  Create a virtual environment: `uv venv`
3.  Activate the environment: `source .venv/bin/activate` (or `.venv\Scripts\activate` on Windows).
4.  Install all dependencies (including CrabNet): `uv sync`

## Overview

**Data Requirements:**
- **Datasets:** All data for this assignment is located in the `HW3/data/` directory.
- The microstructure images for Part 1 are in the `HW3/data/micrographs/` sub-folder.
- The crossed-barrel simulation data for Part 1 (Task 1.9) is `HW3/data/crossed_barrel_dataset.csv`.
- The RRUFF powder XRD data for Part 2 is in `HW3/data/powderXRD/XY_Processed.zip`. **Manually unzip** this file into `HW3/data/powderXRD/XY_Processed/` before starting Part 2.
- The shear modulus dataset for Part 3 is `HW3/data/shearmodulus_aflow.csv`.
- The heat capacity dataset for Part 3 (Task 3.7) is `HW3/data/cp_data_cleaned.csv`.

**Estimated Time (Senior UG / Grad):**
- Part 1: ~2.5-3.5 hours
- Part 2: ~3-4.5 hours
- Part 3: ~2-3 hours
- Total: ~7.5-11 hours

The goal of this assignment is to implement and train deep learning models for materials science applications, including MLP surrogate models, VAE, RNNs, GANs, and transformer-based architectures.

## Part 1: Variational Autoencoder (VAE)

### Task 1.1: Import Libraries

In [ ]:
# Your code here

### Task 1.2: Load Images
Define the `load_images` function and load the image paths from the `data/micrographs` folder.

In [ ]:
# Your code here

### Task 1.3: Create MicrographDataset Class

In [ ]:
# Your code here

### Task 1.4: Implement Convolutional VAE

In [ ]:
# Your code here

### Task 1.5: Define VAE Loss Function

In [ ]:
# Your code here

### Task 1.6: Create Training Function

In [ ]:
# Your code here

### Task 1.7: Train the VAE

In [ ]:
# Your code here

### Task 1.8: Generate Similar Images

In [ ]:
# Your code here

### Task 1.9: MLP Surrogate Model for Crossed-Barrel Toughness
Load `data/crossed_barrel_dataset.csv` and train a feedforward neural network to predict toughness from the four geometric parameters (n, theta, r, t).

**Requirements:**
- Normalize all input features
- Split into train/test (80/20)
- Implement an MLP with at least 2 hidden layers
- Report MSE, MAE, and R² on the test set
- Save parity plot as `crossedbarrel_parity_plot.png`
- Save training loss as `crossedbarrel_training_loss.png`
- Save model as `crossedbarrel_mlp_model.pth`

**Default hyperparameters:** hidden layers = [64, 128, 64], ReLU, epochs = 100, lr = 1e-3, batch_size = 32

In [ ]:
# Your code here


## Part 2: RRUFF Powder XRD Sequence Models (GAN, RNN, Transformer Decoder)


### Task 2.1: Load and Parse RRUFF XRD Files
- Manually unzip `data/powderXRD/XY_Processed.zip` into `data/powderXRD/XY_Processed/`
- Read `.txt` files inside `data/powderXRD/XY_Processed/`
- Parse mineral name from the filename (substring before the first `__`)
- Ignore header lines that start with `##`
- Read the two numeric columns as `2theta` and `intensity`

**Function signature:** `load_xrd_patterns(folder_path)` — returns a list of `(mineral_name, theta, intensity)` records

In [ ]:
# Your code here


### Task 2.2: Preprocess and Build a Dataset
- Resample all patterns to a fixed grid (e.g., 5-80 degrees with uniform step)
- Normalize intensities to [0,1]
- Keep only minerals with at least 5 patterns
- Select at least 10 mineral classes
- Create train/validation/test splits with class balance

**Default preprocessing (use unless you justify changes):**
- 2theta grid: 5.0 to 80.0 degrees, step = 0.1 (751 points)


In [ ]:
# Your code here


### Task 2.3: GAN for Synthetic XRD Augmentation

**Step 1 — Establish a baseline:** Before applying any augmentation, implement the RNN classifier from Task 2.4 and train it on the un-augmented training set. Record its test-set macro-F1 as your baseline.

**Step 2 — GAN augmentation:**
- **Define minority classes** as those with fewer samples than the median class count in the training set.
- Train a 1D GAN on minority classes to generate enough synthetic patterns to bring each up to at least the median count.
- Retrain the RNN classifier on the augmented training set.
- Report macro-F1 before (baseline) and after augmentation.

**Default hyperparameters (use unless you justify changes):**
- latent_dim = 32
- generator hidden sizes = [256, 512]
- discriminator hidden sizes = [512, 256]
- batch_size = 64
- epochs = 50
- learning_rate = 2e-4 (Adam), betas = (0.5, 0.999)

**Architecture guidance (1D MLP GAN):**
- Generator: latent_dim -> 256 -> 512 -> seq_len (ReLU, final Sigmoid)
- Discriminator: seq_len -> 512 -> 256 -> 1 (LeakyReLU(0.2), final Sigmoid)

In [ ]:
# Your code here


### Task 2.4: RNN Classifier (Using Augmented Data)
- Train an LSTM or GRU classifier
- Report accuracy and macro-F1 on the test set
- Plot training loss and validation accuracy
- Produce a confusion matrix

**Default hyperparameters (use unless you justify changes):**
- model = GRU
- input_size = 1
- hidden_size = 128
- num_layers = 1
- dropout = 0.2
- batch_size = 64
- epochs = 20
- learning_rate = 1e-3 (Adam)

**Architecture guidance (RNN classifier):**
- Input tensor shape: (batch, seq_len, 1)
- Use last hidden state for classification
- Classifier head: hidden_size -> num_classes


In [ ]:
# Your code here


### Task 2.5: Decoder-Only Transformer for XRD Generation
- Use causal masking for autoregressive prediction
- Report test MSE for next-step prediction
- Generate a full synthetic pattern from a short seed

**Default hyperparameters (use unless you justify changes):**
- d_model = 128
- nhead = 4
- num_layers = 2
- dim_feedforward = 256
- dropout = 0.1
- batch_size = 64
- epochs = 20
- learning_rate = 1e-3 (Adam)

**Architecture guidance (decoder-only transformer):**
- Input tensor shape: (batch, seq_len, 1) projected to d_model
- Use causal mask so position t cannot see positions > t
- Output head: d_model -> 1 to predict next-step intensity

**Autoregressive generation tips:**
- Train with teacher forcing on shifted sequences: x_in = x[:, :-1], x_target = x[:, 1:]
- At generation time, start with a short seed (e.g., first 20 points)
- Iteratively append predictions to build a full-length sequence


In [ ]:
# Your code here


### Task 2.6: Comparison and Improvement Questions
Answer the following questions (use metrics different from Tasks 2.4 and 2.5):
1. What is your balanced accuracy and weighted F1 for the RNN classifier?
2. After GAN augmentation, what is the new balanced accuracy? Did it improve?
3. For the transformer, report MAE for next-step prediction (do not reuse MSE).
4. What concrete improvement did you achieve by changing a hyperparameter or architecture (what changed and what metric improved)?


In [ ]:
# Your response here


## Part 3: CrabNet Transformer Model


### Task 3.1: Setup CrabNet
Load the shear modulus dataset from `data/shearmodulus_aflow.csv`. The file already has the required `formula` and `target` columns.

In [ ]:
# Your code here


### Task 3.2: Prepare Data


In [ ]:
# Your code here


### Task 3.3: Train CrabNet Model
- Train the model on the training set
- Track the training loss for each epoch
- Plot the loss curve

**Default hyperparameters (use unless you justify changes):**
- epochs = 50
- batch_size = 128
- learning_rate = 1e-3
- weight_decay = 1e-5


In [ ]:
# Your code here


### Task 3.4: Evaluate on Test Set


In [ ]:
# Your code here


### Task 3.5: Predict on New Materials


In [ ]:
# Your code here


### Task 3.6: Improvement Questions
Answer the following questions (use a metric different from the MSE in Task 3.4):
1. What metric did you use (e.g., MAE or R^2)?
2. What was the baseline value before changes?
3. What hyperparameter or architectural change did you make?
4. What was the new metric after the change?


In [ ]:
# Your response here


### Task 3.7: CrabNet for Heat Capacity Prediction
Filter `data/cp_data_cleaned.csv` to T = 298K, rename the `Cp` column to `target`, and train a second CrabNet model to predict room-temperature heat capacity from chemical formula.

**Requirements:**
- Filter to rows where `T == 298.0` (~243 samples)
- Rename `Cp` → `target`, drop the `T` column
- Split into train/test (90/10)
- Train CrabNet (same procedure as Task 3.3)
- Report MAE and R² on the test set
- Compare results to the shear modulus model and briefly discuss any differences
- Save training loss as `crabnet_cp_training_loss.png`
- Save parity plot as `crabnet_cp_parity_plot.png`

In [ ]:
# Your code here
